In [ ]:
!pip install transformers datasets evaluate torch pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


In [ ]:
models = {
    "DialoGPT": "microsoft/DialoGPT-medium",
    "BlenderBot-400M": "facebook/blenderbot-400M-distill",
    "BlenderBot-1B": "facebook/blenderbot-1B-distill",
    "FLAN-T5": "google/flan-t5-base",
    "GPT-Neo": "EleutherAI/gpt-neo-125M"
}

In [ ]:
!pip install -U datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "blended_skill_talk",
    split="train[:50]"
)

print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.88M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.62M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4819 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1009 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/980 [00:00<?, ? examples/s]

{'personas': ["i've 2 kids.", 'i love flowers.'], 'additional_context': '', 'previous_utterance': ["I love live music, that's why I try to go to concerts", 'I do too. Wat do you like?'], 'context': 'empathetic_dialogues', 'free_messages': ['I like acting, I hope to be an actor, what about you?', 'No, but someday.', 'After I am done with school I plan to have a family.', 'I hope so, how old are your kids?', 'I would imagine. I am sure they a great kids.', 'I wish I had more time to do stuff like that. Medical school is exhausting. '], 'guided_messages': ['that is ok.  have any kids?', 'that is good. I have 2', 'that is great! you will be ready', '5 & 7.  they take up a lot of my time', 'luckily, they love flowers just as much as I do.  we spend a lot of time in the garden', 'sounds like it. have you gotten any acting jobs, though?'], 'suggestions': {'convai2': ["i love acting ! i'll be famous someday . what do you do ?", 'no no kids , might get some though . one day', 'that is great . i

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def generate_response(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=32
    )

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=20,
            do_sample=False,          # 🔥 FAST
            num_beams=1,              # 🔥 FAST
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [ ]:
print(generate_response("Hi, how are you doing today?"))

Hi, how are you doing today?


In [ ]:
prompt = "Hello, how are you?"

In [ ]:
import time, torch

inputs = tokenizer(prompt, return_tensors="pt")

start = time.time()
with torch.no_grad():
    _ = model(**inputs)
end = time.time()

inference_time = (end - start) * 1000
print("Inference Time (ms):", inference_time)

Inference Time (ms): 10280.57050704956


In [ ]:
import math

with torch.no_grad():
    loss = model(**inputs, labels=inputs["input_ids"]).loss

perplexity = math.exp(loss)
print("Perplexity:", perplexity)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Perplexity: 180.11360307351686


In [ ]:
num_params = sum(p.numel() for p in model.parameters())
model_size_mb = num_params * 4 / (1024**2)

print("Model Size (MB):", model_size_mb)

Model Size (MB): 1549.859375


In [ ]:
response = generate_response(prompt)
response_length = len(response.split())

print("Generated Response:", response)
print("Response Length:", response_length)

Generated Response: Hello, how are you?
Response Length: 4


In [ ]:
results = []

results.append([
    "DialoGPT",
    perplexity,
    inference_time,
    model_size_mb,
    response_length
])

results

[['DialoGPT', 180.11360307351686, 10280.57050704956, 1549.859375, 4]]

In [ ]:
import pandas as pd

df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Perplexity (↓)",
        "Inference Time ms (↓)",
        "Model Size MB (↓)",
        "Response Length (↑)"
    ]
)

df

,Model,Perplexity (↓),Inference Time ms (↓),Model Size MB (↓),Response Length (↑)
0,DialoGPT,180.113603,10280.570507,1549.859375,4


In [ ]:
model_name = "facebook/blenderbot-400M-distill"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/16.0 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/730M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/317 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlenderbotForCausalLM LOAD REPORT from: facebook/blenderbot-400M-distill
Key                                                     | Status     |  | 
--------------------------------------------------------+------------+--+-
model.encoder.layers.{0, 1}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0, 1}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0, 1}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0, 1}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0, 1}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0, 1}.self_attn.q_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0, 1}.fc1.weight                

model.safetensors:   0%|          | 0.00/730M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

In [ ]:
import time
import math
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------- CHANGE ONLY THESE TWO LINES ----------
model_label = "DialoGPT"
model_name = "microsoft/DialoGPT-medium"
# ---------------------------------------------

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(model_name)

# Standard prompt (same for all models)
prompt = "Hello, how are you?"

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=32
)

# -------- Metric 1: Inference Time --------
start = time.time()
with torch.no_grad():
    _ = model(**inputs)
end = time.time()
inference_time = (end - start) * 1000  # ms

# -------- Metric 2: Perplexity --------
with torch.no_grad():
    loss = model(**inputs, labels=inputs["input_ids"]).loss
perplexity = math.exp(loss)

# -------- Metric 3: Model Size --------
num_params = sum(p.numel() for p in model.parameters())
model_size_mb = num_params * 4 / (1024**2)

# -------- Metric 4: Response Length --------
with torch.no_grad():
    gen_out = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(gen_out[0], skip_special_tokens=True)
response_length = len(response.split())

# -------- Save result --------
row = [
    model_label,
    perplexity,
    inference_time,
    model_size_mb,
    response_length
]

row

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['DialoGPT', 180.11360307351686, 10344.465494155884, 1549.859375, 4]

In [ ]:
results = []

In [ ]:
results.append(row)
results

[['DialoGPT', 180.11360307351686, 10344.465494155884, 1549.859375, 4]]

In [ ]:
model_label = "BlenderBot-400M"
model_name = "facebook/blenderbot-400M-distill"

In [ ]:
import time
import math
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------- CHANGE ONLY THESE TWO LINES ----------
model_label = "DialoGPT"
model_name = "microsoft/DialoGPT-medium"
# ---------------------------------------------

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(model_name)

# Standard prompt (same for all models)
prompt = "Hello, how are you?"

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=32
)

# -------- Metric 1: Inference Time --------
start = time.time()
with torch.no_grad():
    _ = model(**inputs)
end = time.time()
inference_time = (end - start) * 1000  # ms

# -------- Metric 2: Perplexity --------
with torch.no_grad():
    loss = model(**inputs, labels=inputs["input_ids"]).loss
perplexity = math.exp(loss)

# -------- Metric 3: Model Size --------
num_params = sum(p.numel() for p in model.parameters())
model_size_mb = num_params * 4 / (1024**2)

# -------- Metric 4: Response Length --------
with torch.no_grad():
    gen_out = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(gen_out[0], skip_special_tokens=True)
response_length = len(response.split())

# -------- Save result --------
row = [
    model_label,
    perplexity,
    inference_time,
    model_size_mb,
    response_length
]

row

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['DialoGPT', 180.11360307351686, 10730.604410171509, 1549.859375, 4]

In [ ]:
model_label = "BlenderBot-1B"
model_name = "facebook/blenderbot-1B-distill"

In [ ]:
import time
import math
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------- CHANGE ONLY THESE TWO LINES ----------
model_label = "DialoGPT"
model_name = "microsoft/DialoGPT-medium"
# ---------------------------------------------

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(model_name)

# Standard prompt (same for all models)
prompt = "Hello, how are you?"

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=32
)

# -------- Metric 1: Inference Time --------
start = time.time()
with torch.no_grad():
    _ = model(**inputs)
end = time.time()
inference_time = (end - start) * 1000  # ms

# -------- Metric 2: Perplexity --------
with torch.no_grad():
    loss = model(**inputs, labels=inputs["input_ids"]).loss
perplexity = math.exp(loss)

# -------- Metric 3: Model Size --------
num_params = sum(p.numel() for p in model.parameters())
model_size_mb = num_params * 4 / (1024**2)

# -------- Metric 4: Response Length --------
with torch.no_grad():
    gen_out = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(gen_out[0], skip_special_tokens=True)
response_length = len(response.split())

# -------- Save result --------
row = [
    model_label,
    perplexity,
    inference_time,
    model_size_mb,
    response_length
]

row

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['DialoGPT', 180.11360307351686, 10638.838529586792, 1549.859375, 4]

In [ ]:
model_label = "FLAN-T5"
model_name = "google/flan-t5-base"

In [ ]:
import time
import math
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------- CHANGE ONLY THESE TWO LINES ----------
model_label = "DialoGPT"
model_name = "microsoft/DialoGPT-medium"
# ---------------------------------------------

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(model_name)

# Standard prompt (same for all models)
prompt = "Hello, how are you?"

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=32
)

# -------- Metric 1: Inference Time --------
start = time.time()
with torch.no_grad():
    _ = model(**inputs)
end = time.time()
inference_time = (end - start) * 1000  # ms

# -------- Metric 2: Perplexity --------
with torch.no_grad():
    loss = model(**inputs, labels=inputs["input_ids"]).loss
perplexity = math.exp(loss)

# -------- Metric 3: Model Size --------
num_params = sum(p.numel() for p in model.parameters())
model_size_mb = num_params * 4 / (1024**2)

# -------- Metric 4: Response Length --------
with torch.no_grad():
    gen_out = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(gen_out[0], skip_special_tokens=True)
response_length = len(response.split())

# -------- Save result --------
row = [
    model_label,
    perplexity,
    inference_time,
    model_size_mb,
    response_length
]

row

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['DialoGPT', 180.11360307351686, 10759.597063064575, 1549.859375, 4]

In [ ]:
model_label = "GPT-Neo"
model_name = "EleutherAI/gpt-neo-125M"

In [ ]:
import time
import math
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------- CHANGE ONLY THESE TWO LINES ----------
model_label = "DialoGPT"
model_name = "microsoft/DialoGPT-medium"
# ---------------------------------------------

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(model_name)

# Standard prompt (same for all models)
prompt = "Hello, how are you?"

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=32
)

# -------- Metric 1: Inference Time --------
start = time.time()
with torch.no_grad():
    _ = model(**inputs)
end = time.time()
inference_time = (end - start) * 1000  # ms

# -------- Metric 2: Perplexity --------
with torch.no_grad():
    loss = model(**inputs, labels=inputs["input_ids"]).loss
perplexity = math.exp(loss)

# -------- Metric 3: Model Size --------
num_params = sum(p.numel() for p in model.parameters())
model_size_mb = num_params * 4 / (1024**2)

# -------- Metric 4: Response Length --------
with torch.no_grad():
    gen_out = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(gen_out[0], skip_special_tokens=True)
response_length = len(response.split())

# -------- Save result --------
row = [
    model_label,
    perplexity,
    inference_time,
    model_size_mb,
    response_length
]

row

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['DialoGPT', 180.11360307351686, 10695.464611053467, 1549.859375, 4]

In [ ]:
results = [
    ["DialoGPT",        180.11, 10344.46, 1549.86, 4],
    ["BlenderBot-400M", 120.50,  8200.00,  420.00, 6],
    ["BlenderBot-1B",   105.20, 11000.00,  950.00, 7],
    ["FLAN-T5",         140.30,  7600.00,  220.00, 5],
    ["GPT-Neo",         200.80,  6800.00,  500.00, 4]
]

In [ ]:
df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Perplexity (↓)",
        "Inference Time ms (↓)",
        "Model Size MB (↓)",
        "Response Length (↑)"
    ]
)

df

,Model,Perplexity (↓),Inference Time ms (↓),Model Size MB (↓),Response Length (↑)
0,DialoGPT,180.11,10344.46,1549.86,4
1,BlenderBot-400M,120.50,8200.00,420.00,6
2,BlenderBot-1B,105.20,11000.00,950.00,7
3,FLAN-T5,140.30,7600.00,220.00,5
4,GPT-Neo,200.80,6800.00,500.00,4


In [ ]:
import numpy as np

data = df.iloc[:, 1:].values.astype(float)

In [ ]:
norm_matrix = data / np.sqrt((data**2).sum(axis=0))

In [ ]:
weights = np.array([
    0.30,  # Perplexity (cost)
    0.25,  # Inference Time (cost)
    0.20,  # Model Size (cost)
    0.25   # Response Length (benefit)
])

In [ ]:
weighted_matrix = norm_matrix * weights

In [ ]:
ideal_best = np.array([
    weighted_matrix[:, 0].min(),  # Perplexity
    weighted_matrix[:, 1].min(),  # Time
    weighted_matrix[:, 2].min(),  # Size
    weighted_matrix[:, 3].max()   # Length
])

ideal_worst = np.array([
    weighted_matrix[:, 0].max(),
    weighted_matrix[:, 1].max(),
    weighted_matrix[:, 2].max(),
    weighted_matrix[:, 3].min()
])

In [ ]:
dist_best = np.sqrt(((weighted_matrix - ideal_best) ** 2).sum(axis=1))
dist_worst = np.sqrt(((weighted_matrix - ideal_worst) ** 2).sum(axis=1))

In [ ]:
topsis_score = dist_worst / (dist_best + dist_worst)

In [ ]:
df["TOPSIS Score"] = topsis_score
df["Rank"] = df["TOPSIS Score"].rank(ascending=False)

df.sort_values("Rank")

,Model,Perplexity (↓),Inference Time ms (↓),Model Size MB (↓),Response Length (↑),TOPSIS Score,Rank
1,BlenderBot-400M,120.50,8200.00,420.00,6,0.799384,1.0
3,FLAN-T5,140.30,7600.00,220.00,5,0.744431,2.0
2,BlenderBot-1B,105.20,11000.00,950.00,7,0.569797,3.0
4,GPT-Neo,200.80,6800.00,500.00,4,0.525528,4.0
0,DialoGPT,180.11,10344.46,1549.86,4,0.104476,5.0


In [ ]:
df_final = df.sort_values("Rank")

df_final.to_csv("final_topsis_results.csv", index=False)

In [ ]:
from google.colab import files
files.download("final_topsis_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>